In [1]:
from pathlib import Path
import jupyter_black
import numpy as np
from tqdm import tqdm


jupyter_black.load()
import os

# Source - https://stackoverflow.com/a/36835741
# Posted by mbh86
# Retrieved 2026-09-10, License - CC BY-SA 3.0

from IPython.core.interactiveshell import InteractiveShell

InteractiveShell.ast_node_interactivity = "all"

In [2]:
os.getcwd()
members = list(range(1, 41))
print(members)

'/gws/ssde/j25b/canari/users/jonnyhtw/canari-CF-aggregation/priority-variables/additional/fill-gaps'

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40]


In [3]:
archive_location = "/badc/canari/data/priority"
os.listdir(archive_location)

['HIST2', 'SSP370', '00README_catalogue_and_licence.txt']

In [4]:
gws_location = "/gws/ssde/j25b/canari/public/priority-variables/data"
os.listdir(gws_location)

['HIST2', 'SSP370']

In [62]:
for i in members:
    archive_path = Path(f"{archive_location}/HIST2/{i:02d}")
    gws_path = Path(f"{gws_location}/HIST2/{i}")

    archive_count = sum(1 for p in archive_path.rglob("*.nc") if p.is_file())
    gws_count = sum(1 for p in gws_path.rglob("*.nc") if p.is_symlink())

    print(f"Member {i:02d} -> Archive: {archive_count} | GWS: {gws_count}")

Member 01 -> Archive: 14105 | GWS: 14105
Member 02 -> Archive: 14105 | GWS: 14105
Member 03 -> Archive: 14105 | GWS: 14105
Member 04 -> Archive: 14105 | GWS: 14105
Member 05 -> Archive: 14105 | GWS: 14105
Member 06 -> Archive: 14105 | GWS: 14105
Member 07 -> Archive: 14105 | GWS: 14105
Member 08 -> Archive: 14105 | GWS: 14105
Member 09 -> Archive: 14105 | GWS: 14105
Member 10 -> Archive: 14105 | GWS: 14105
Member 11 -> Archive: 14105 | GWS: 14105
Member 12 -> Archive: 14105 | GWS: 14105
Member 13 -> Archive: 14105 | GWS: 14105
Member 14 -> Archive: 14105 | GWS: 14105
Member 15 -> Archive: 14105 | GWS: 14105
Member 16 -> Archive: 14105 | GWS: 14105
Member 17 -> Archive: 14105 | GWS: 14105
Member 18 -> Archive: 14082 | GWS: 14082
Member 19 -> Archive: 14105 | GWS: 14105
Member 20 -> Archive: 14105 | GWS: 14105
Member 21 -> Archive: 14053 | GWS: 14053
Member 22 -> Archive: 14105 | GWS: 14105
Member 23 -> Archive: 14105 | GWS: 14105
Member 24 -> Archive: 14105 | GWS: 14105
Member 25 -> Arc

So for member 18, there are fewer files than expected. This is because some are missing from the GWS. Let's find out which ones they are...

In [13]:
members = [1, 18, 21, 29, 31, *range(36, 41)]

results = {i: {"gws": []} for i in members}

for i in tqdm(members, desc="Auditing climate datasets"):
    gws_path = Path(f"{gws_location}/HIST2/{i}")

    # GWS Symlinks
    results[i]["gws"] = sorted(str(p) for p in gws_path.rglob("*.nc") if p.is_symlink())

Auditing climate datasets: 100%|██████████| 10/10 [00:28<00:00,  2.81s/it]


In [14]:
# Clean relative paths using member ID dynamically
set_gws_1 = {
    (sub := p.split("HIST2/")[1].split("/", 2)[2]).rsplit("/", 1)[0]
    + "/"
    + sub.rsplit("/", 1)[1][sub.rsplit("/", 1)[1].find("_") - 1 :].replace("_1_", "_")
    for p in results[1]["gws"]
}

set_gws_18 = {
    (sub := p.split("HIST2/")[1].split("/", 2)[2]).rsplit("/", 1)[0]
    + "/"
    + sub.rsplit("/", 1)[1][sub.rsplit("/", 1)[1].find("_") - 1 :].replace("_18_", "_")
    for p in results[18]["gws"]
}

# Clean, normalized missing signatures
missing_in_18 = sorted(list(set_gws_1 - set_gws_18))

In [53]:
members = [1, 18, 21, 29, 31, *range(36, 41)]

# 1. Map Member 1's normalized signatures to its raw paths & extract Member 1's run ID
gws_1_map = {
    (sub := p.split("HIST2/")[1].split("/", 2)[2]).rsplit("/", 1)[0]
    + "/"
    + sub.rsplit("/", 1)[1][sub.rsplit("/", 1)[1].find("_") - 1 :].replace(
        "_1_", "_"
    ): p
    for p in results[1]["gws"]
}
run_id_1 = results[1]["gws"][0].rsplit("/", 1)[1].split("_")[0][:-1]

# 2. Extract run IDs and normalized signature sets for all other target members
run_ids = {
    m: results[m]["gws"][0].rsplit("/", 1)[1].split("_")[0][:-1]
    for m in members
    if m != 1
}

gws_signatures = {
    m: {
        (sub := p.split("HIST2/")[1].split("/", 2)[2]).rsplit("/", 1)[0]
        + "/"
        + sub.rsplit("/", 1)[1][sub.rsplit("/", 1)[1].find("_") - 1 :].replace(
            f"_{m}_", "_"
        )
        for p in results[m]["gws"]
    }
    for m in members
    if m != 1
}

# 3. Construct expected paths using each member's specific run ID and member ID
missing_files_expected = {
    m: sorted(
        gws_1_map[sig]
        .replace("/HIST2/1/", f"/HIST2/{m}/")
        .replace(f"/{run_id_1}", f"/{run_ids[m]}")
        .replace("_1_", f"_{m}_")
        for sig in (set(gws_1_map) - gws_signatures[m])
    )
    for m in members
    if m != 1
}

# 4. Write missing paths to file cleanly (voiding file.write character count returns)
with open("missing_expected_gws_files.txt", "w") as f:
    _ = f.write("=== MISSING EXPECTED GWS FILES RELATIVE TO MEMBER 1 ===\n\n")
    for member_id, file_paths in missing_files_expected.items():
        _ = f.write(f"--- Member {member_id} ---\n")
        for path in file_paths:
            _ = f.write(f"{path}\n")
        _ = f.write("\n")

# Explicitly end cell with None to suppress any auto-printed return values
None